# Alpha101 Metrics Audit

This notebook reruns the Alpha101 factory and robustness passes from the cached universe panels, then adds rolling Sharpe, rolling volatility, rolling correlation to NIFTY 50, average holding period, and benchmark-relative diagnostics for the final shortlist and the promoted exact-OHLCV queue.

The benchmark source is the local `project_mft.duckdb` index cache populated by the repo data loader. The notebook fails clearly if the cached NIFTY series is missing.


## Run Controls

- Factory and robustness can be refreshed from the latest cached universe panels when you flip the refresh flags.
- The notebook writes its audit tables under `research/artifacts/alpha101_research_factory/alpha101_metrics_audit/`.
- The default execution path uses the cached artifact outputs so the notebook stays runnable in this environment; set the refresh flags to `True` on a larger machine to force a rebuild.
- The detailed metrics recomputation runs for the final shortlist and promoted exact-OHLCV queue.
- The benchmark loader reads the local NIFTY cache first and fails clearly if the cache is missing or empty.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "research/notebooks/alpha_001/research").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
NOTEBOOK_ROOT = REPO_ROOT / "research/notebooks/alpha_001"
for path in (REPO_ROOT, NOTEBOOK_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from research.alpha101_engine import ALPHA101_ARTIFACT_DIR, load_panel
from research.alpha101_factory import run_alpha101_factory
from research.alpha101_metrics_audit import (
    DEFAULT_BENCHMARK_PATH,
    ROLLING_WINDOWS,
    audit_selection_frame,
    build_selection_audit,
    load_benchmark_returns,
)
from research.alpha101_robustness import run_alpha101_robustness, run_alpha101_robustness_batch2

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

RUN_FACTORY_REFRESH = False
RUN_ROBUSTNESS_REFRESH = False
RUN_ROBUSTNESS_BATCH2_REFRESH = False
MAX_WORKERS = 1
COST_BPS = 20.0
ARTIFACT_DIR = ALPHA101_ARTIFACT_DIR / "alpha101_metrics_audit"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR


In [ ]:
benchmark_returns = load_benchmark_returns(DEFAULT_BENCHMARK_PATH)

benchmark_audit = pd.DataFrame(
    [
        {
            "path": str(DEFAULT_BENCHMARK_PATH),
            "sessions": int(len(benchmark_returns)),
            "start": benchmark_returns.index.min(),
            "end": benchmark_returns.index.max(),
            "mean_return": benchmark_returns.mean(),
            "volatility": benchmark_returns.std(ddof=0),
        }
    ]
)
display(benchmark_audit)
display(Markdown(f"Benchmark coverage is {len(benchmark_returns)} sessions across {benchmark_returns.index.min().date()} to {benchmark_returns.index.max().date()}."))


In [ ]:
import json

if RUN_FACTORY_REFRESH or RUN_ROBUSTNESS_REFRESH or RUN_ROBUSTNESS_BATCH2_REFRESH:
    factory_outputs = run_alpha101_factory(
        max_workers=MAX_WORKERS,
        refresh=RUN_FACTORY_REFRESH,
        progress=True,
        reaggregate=RUN_FACTORY_REFRESH,
    )
    robustness_outputs = run_alpha101_robustness(refresh=RUN_ROBUSTNESS_REFRESH, progress=True)
    batch2_outputs = run_alpha101_robustness_batch2(refresh=RUN_ROBUSTNESS_BATCH2_REFRESH, progress=True)
    factory_top = factory_outputs["leaderboard"].sort_values("research_score", ascending=False).head(25)
    robustness_top = robustness_outputs["shortlist"]
    promoted_exact_raw = batch2_outputs["combined_shortlist"]
else:
    snapshot = json.loads((ALPHA101_ARTIFACT_DIR / "alpha101_metrics_snapshot.json").read_text())
    factory_top = pd.DataFrame(snapshot["reports"]["factory"]["top_25"])
    robustness_top = pd.DataFrame(snapshot["reports"]["robustness_batch1"]["top_rows"])
    promoted_exact_raw = pd.DataFrame(snapshot["reports"]["robustness_batch2"]["combined_promoted_exact_ohlcv"])

promoted_filter = promoted_exact_raw["final_status"].eq("promote_to_deeper_research")
if "input_quality_tier" in promoted_exact_raw.columns:
    promoted_filter &= promoted_exact_raw["input_quality_tier"].eq("exact_ohlcv")
promoted_exact_raw = promoted_exact_raw[promoted_filter]
final_shortlist = factory_top.merge(robustness_top, on=["panel", "alpha_id"], how="inner", suffixes=("", "_robustness"))
promoted_exact = factory_top.merge(promoted_exact_raw, on=["panel", "alpha_id"], how="inner", suffixes=("", "_promoted"))
promoted_exact = promoted_exact[promoted_exact["input_quality_tier"].eq("exact_ohlcv")]

factory_cols = ["panel", "alpha_id", "family", "input_quality_tier", "classification", "best_5d_ic", "best_20bps_active_sharpe", "best_mask", "best_signal_transform", "best_strategy", "research_score"]
shortlist_base_cols = ["panel", "alpha_id", "final_status", "median_test_active_sharpe", "median_test_active_cagr", "median_test_rank_ic", "median_turnover", "best_mask", "best_signal_transform", "best_strategy"]
promoted_base_cols = ["panel", "alpha_id", "final_status", "median_test_active_sharpe", "median_test_active_cagr", "median_test_rank_ic", "median_turnover", "best_mask", "best_signal_transform", "best_strategy"]

display(factory_top[factory_cols].head(20))
display(final_shortlist[shortlist_base_cols].sort_values("median_test_active_sharpe", ascending=False).head(20))
display(promoted_exact[promoted_base_cols].sort_values("median_test_active_sharpe", ascending=False).head(20))

shortlist_audit_source = final_shortlist.sort_values("median_test_active_sharpe", ascending=False)
promoted_audit_source = promoted_exact.sort_values("median_test_active_sharpe", ascending=False)
shortlist_audit = audit_selection_frame(shortlist_audit_source, benchmark_returns, COST_BPS)
promoted_audit = audit_selection_frame(promoted_audit_source, benchmark_returns, COST_BPS)

shortlist_audit.to_csv(ARTIFACT_DIR / "alpha101_metrics_audit_shortlist.csv", index=False)
promoted_audit.to_csv(ARTIFACT_DIR / "alpha101_metrics_audit_promoted_exact_ohlcv.csv", index=False)

shortlist_cols = [
    "panel", "alpha_id", "final_status", "median_test_active_sharpe", "median_test_active_cagr", "median_test_rank_ic", "median_turnover",
    "strategy_cagr", "strategy_sharpe", "strategy_sortino", "strategy_max_drawdown", "strategy_avg_daily_turnover",
    "average_holding_period", "trade_count", "beta_to_nifty50", "information_ratio", "rolling_sharpe_21", "rolling_vol_21", "rolling_corr_21",
]
promoted_cols = [
    "panel", "alpha_id", "final_status", "median_test_active_sharpe", "median_test_active_cagr", "median_test_rank_ic", "median_turnover",
    "strategy_cagr", "strategy_sharpe", "strategy_sortino", "strategy_max_drawdown", "strategy_avg_daily_turnover",
    "average_holding_period", "trade_count", "beta_to_nifty50", "information_ratio", "rolling_sharpe_21", "rolling_vol_21", "rolling_corr_21",
]
display(shortlist_audit.sort_values("median_test_active_sharpe", ascending=False)[shortlist_cols])
display(promoted_audit.sort_values("median_test_active_sharpe", ascending=False)[promoted_cols])


In [ ]:
# The detailed snapshot recovery and audit run in the previous cell.
pass


In [ ]:
def plot_audit_bundle(audit: dict[str, object], title: str) -> None:
    rolling_strategy = audit["rolling_strategy"]
    rolling_relative = audit["rolling_relative"]
    summary = audit["summary"]
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    for window in ROLLING_WINDOWS:
        axes[0].plot(rolling_strategy.index, rolling_strategy[f"rolling_sharpe_{window}"], label=f"{window}d")
        axes[1].plot(rolling_strategy.index, rolling_strategy[f"rolling_vol_{window}"], label=f"{window}d")
        axes[2].plot(rolling_relative.index, rolling_relative[f"rolling_corr_{window}"], label=f"{window}d")
    axes[0].set_title(title)
    axes[0].set_ylabel("Sharpe")
    axes[1].set_ylabel("Vol")
    axes[2].set_ylabel("Corr to NIFTY")
    axes[2].set_xlabel("Date")
    for axis in axes:
        axis.grid(True, alpha=0.25)
        axis.legend(loc="upper left")
    fig.suptitle(f"{summary['alpha_id']} | {summary['panel']} | {summary.get('final_status', 'n/a')}")
    fig.tight_layout()
    display(fig)
    plt.close(fig)

shortlist_plot_audits = [
    build_selection_audit(row, benchmark_returns, COST_BPS)
    for _, row in shortlist_audit.sort_values("median_test_active_sharpe", ascending=False).head(2).iterrows()
]
promoted_plot_audits = [
    build_selection_audit(row, benchmark_returns, COST_BPS)
    for _, row in promoted_audit.head(2).iterrows()
]

for audit in shortlist_plot_audits:
    plot_audit_bundle(audit, f"Final shortlist rolling diagnostics: {audit['summary']['alpha_id']}")
for audit in promoted_plot_audits:
    plot_audit_bundle(audit, f"Promoted exact-OHLCV rolling diagnostics: {audit['summary']['alpha_id']}")


## Notes

- Rolling Sharpe and rolling volatility are computed on the selected strategy return stream.
- Rolling correlation and beta are benchmark-relative against the cached NIFTY 50 series.
- Average holding period is measured from contiguous non-zero position streaks in the selected weight history.
- Trade count is the number of weight state changes across the selected position history.
